In [7]:
!pip install -q yfinance

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from models.wmt_model import WMTTradingModel
from models.nvda_model import NVDATradingModel
from models.mpc_model import MPCTradingModel
from models.xom_model import XOMTradingModel
from models.VAR_model import VARTradingModel

from utils import ForecastingMetrics, TradingMetrics, PortfolioEvaluator
from backtest import Backtest
import yfinance as yf
import numpy as np
import pandas as pd


from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import plotting
import matplotlib.pyplot as plt
import pandas as pd
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage

The best models for each stock are:
- WMT: VAR(0)
    - Provided lower error, higher robustness, and simpler logic
- MPC: ARIMA/ARIMA-GARCH
    - Captured both trend and shock-adjustment behavior
    - Although the MSE is slightly lower than the Naive Baseline, it has higher directional accuracy.
- NVDA: VAR(0)
    - This stock is very volatile 
    - More event-driven rather than dependent on historical data so 
        - enforcing lag-based structure increased forecast error 
    - The mean forecast (VAR(0)) minimized cumulative prediction error better than models trying to impose structure where none existed
- XOM: VAR(1)
    - Provided the best balance between low forecast error, directional accuracy, and alignment

In [3]:
# ChatGPT made this dictionary
stock_categories = {
    "AAPL": "Technology", "MSFT": "Technology", "NVDA": "Technology",
    "AMZN": "Technology", "GOOGL": "Technology", "META": "Technology", "TSLA": "Technology",
    "JPM": "Finance", "BAC": "Finance", "WFC": "Finance", "C": "Finance", "GS": "Finance", "MS": "Finance",
    "KO": "Consumer Goods", "PG": "Consumer Goods", "PEP": "Consumer Goods",
    "WMT": "Consumer Goods", "COST": "Consumer Goods",
    "CL": "Consumer Goods", "XOM": "Energy", "CVX": "Energy", "COP": "Energy",
    "SLB": "Energy", "EOG": "Energy", "MPC": "Energy",
    "SPY": "ETF", "QQQ": "ETF", "DIA": "ETF", "IWM": "ETF", "VTI": "ETF"
}

# Get the inverse of stock_categories
category_stock = {}
for stock, category in stock_categories.items():
    if category not in category_stock:
        category_stock[category] = []
    category_stock[category].append(stock)

In [5]:
target_stock = 'WMT'
related_stocks = category_stock[stock_categories[target_stock]]
wmt_model = VARTradingModel(target_stock, related_stocks)

In [6]:
data = yf.download(related_stocks, start=start, end=end, auto_adjust=True)
lookback_days = 252
closes = data["Close"].dropna().tail(lookback_days)
#closes = closes.asfreq('B')


NameError: name 'start' is not defined

In [7]:
target_stock = 'MPC'
related_stocks = category_stock[stock_categories[target_stock]]
xom_model = VARTradingModel(target_stock, related_stocks)

start = "2022-01-01"
end = "2025-01-01"

data = yf.download(related_stocks, start=start, end=end, auto_adjust=True)
lookback_days = 252
closes = data["Close"]
#closes = closes.asfreq('B')


[*********************100%***********************]  6 of 6 completed


In [8]:
xom_model._best_lag

In [9]:
np.log(closes)

Ticker,COP,CVX,EOG,MPC,SLB,XOM
Date,,,,,,
2022-01-03,4.154185,4.619886,4.328669,4.093457,3.374877,4.009608
2022-01-04,4.196649,4.637918,4.373607,4.125676,3.422285,4.046531
2022-01-05,4.179350,4.644402,4.355083,4.131998,3.422285,4.058892
2022-01-06,4.216204,4.652876,4.375388,4.159603,3.445760,4.082141
2022-01-07,4.243228,4.667133,4.402652,4.170660,3.474135,4.090304
...,...,...,...,...,...,...
2024-12-24,4.541836,4.923387,4.760387,4.885902,3.607156,4.631162
2024-12-26,4.539568,4.924360,4.756905,4.886198,3.607156,4.632007
2024-12-27,4.539877,4.924499,4.756822,4.888264,3.609010,4.631913


In [10]:
xom_model.fit(np.log(closes))
xom_model.forecast(60)

0


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


array([4.91719664, 4.91829058, 4.91938453, 4.92047847, 4.92157241,
       4.92266636, 4.9237603 , 4.92485424, 4.92594819, 4.92704213,
       4.92813608, 4.92923002, 4.93032396, 4.93141791, 4.93251185,
       4.93360579, 4.93469974, 4.93579368, 4.93688763, 4.93798157,
       4.93907551, 4.94016946, 4.9412634 , 4.94235734, 4.94345129,
       4.94454523, 4.94563918, 4.94673312, 4.94782706, 4.94892101,
       4.95001495, 4.95110889, 4.95220284, 4.95329678, 4.95439073,
       4.95548467, 4.95657861, 4.95767256, 4.9587665 , 4.95986044,
       4.96095439, 4.96204833, 4.96314228, 4.96423622, 4.96533016,
       4.96642411, 4.96751805, 4.96861199, 4.96970594, 4.97079988,
       4.97189383, 4.97298777, 4.97408171, 4.97517566, 4.9762696 ,
       4.97736354, 4.97845749, 4.97955143, 4.98064538, 4.98173932])

In [39]:
wmt_model.fit(np.log(closes))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [12]:
wmt_model.forecast(60)

array([4.58145799, 4.58380726, 4.58615653, 4.5885058 , 4.59085507,
       4.59320434, 4.59555361, 4.59790288, 4.60025215, 4.60260142,
       4.60495069, 4.60729996, 4.60964923, 4.6119985 , 4.61434777,
       4.61669704, 4.61904631, 4.62139558, 4.62374485, 4.62609412,
       4.62844339, 4.63079266, 4.63314193, 4.63549119, 4.63784046,
       4.64018973, 4.642539  , 4.64488827, 4.64723754, 4.64958681,
       4.65193608, 4.65428535, 4.65663462, 4.65898389, 4.66133316,
       4.66368243, 4.6660317 , 4.66838097, 4.67073024, 4.67307951,
       4.67542878, 4.67777805, 4.68012732, 4.68247659, 4.68482586,
       4.68717513, 4.6895244 , 4.69187366, 4.69422293, 4.6965722 ,
       4.69892147, 4.70127074, 4.70362001, 4.70596928, 4.70831855,
       4.71066782, 4.71301709, 4.71536636, 4.71771563, 4.7200649 ])

In [59]:
horizon = 60
dummy_df = pd.DataFrame(np.zeros((horizon, len(wmt_model.related_stocks))),
                        columns=wmt_model.related_stocks)
forecast = wmt_model.predict(dummy_df)

In [64]:
forecast

array([4.58145799, 4.58380726, 4.58615653, 4.5885058 , 4.59085507,
       4.59320434, 4.59555361, 4.59790288, 4.60025215, 4.60260142,
       4.60495069, 4.60729996, 4.60964923, 4.61199849, 4.61434776,
       4.61669703, 4.6190463 , 4.62139557, 4.62374484, 4.62609411,
       4.62844338, 4.63079265, 4.63314192, 4.63549119, 4.63784046,
       4.64018973, 4.642539  , 4.64488827, 4.64723753, 4.6495868 ,
       4.65193607, 4.65428534, 4.65663461, 4.65898388, 4.66133315,
       4.66368242, 4.66603169, 4.66838096, 4.67073023, 4.6730795 ,
       4.67542877, 4.67777804, 4.6801273 , 4.68247657, 4.68482584,
       4.68717511, 4.68952438, 4.69187365, 4.69422292, 4.69657219,
       4.69892146, 4.70127073, 4.70362   , 4.70596927, 4.70831854,
       4.71066781, 4.71301708, 4.71536634, 4.71771561, 4.72006488])

In [7]:
def forecast_and_markowitz(as_of_date: str,
                           horizon: int = 60,
                           lookback_days: int = 252,
                           objective: str = "gmir"
                           ) -> tuple[pd.DataFrame, pd.Series]:
    """
    For any date after Jan 2025, re-optimize the portfolio using
    forecasted returns from the best model per stock.

    Parameters
    ----------
    as_of_date : str
        Date string "YYYY-MM-DD". Use any date >= "2025-01-01".
        All data strictly before this date is used for training.
    horizon : int, default=60
        Number of future trading days to forecast.
    lookback_days : int, default=252
        Number of past trading days used for model estimation.
    objective : {"gmir", "gmv"}, default="gmir"
        Markowitz objective: max information ratio or min variance.

    Returns
    -------
    forecast_df : pd.DataFrame
        Shape (horizon, 4) with columns ["WMT", "NVDA", "MPC", "XOM"]
        containing forecasted daily returns.
    weights : pd.Series
        Markowitz optimal weights indexed by ticker, summing to 1.
    """
    tickers = ["WMT", "NVDA", "MPC", "XOM"]
    model_map = {
        "WMT": WMTTradingModel,          # VAR(0) 
        "NVDA": NVDATradingModel,        # VAR(0)
        "MPC": MPCTradingModel,          # ARIMA(2,1,1)/ARIMA-GARCH
        "XOM": XOMTradingModel,          # VAR(1)
    }

    end = pd.to_datetime(as_of_date)
    start = end - pd.tseries.offsets.BDay(int(lookback_days * 1.5))

    forecasts_list: list[np.ndarray] = []

    for t in tickers:
        # 1) download prices up to as_of_date
        data = yf.download(t, start=start, end=end)
        closes = data["Close"].dropna().tail(lookback_days)

        # 2) compute log price
        log_price = np.log(closes).values

        # 3) fit best model and produce 60-day forecast
        model_cls = model_map[t]
        model = model_cls()
        model.fit(log_price)

        dummy_X = np.zeros(horizon)
        fc = np.asarray(model.predict(dummy_X), dtype=float)
        forecasts_list.append(fc)

    # 4) stack forecasts of log prices into (horizon, n_assets) matrix
    log_price_forecast_matrix = np.column_stack(forecasts_list)
    log_price_forecast_df = pd.DataFrame(log_price_forecast_matrix, columns=tickers)

    price_forecast_df = np.exp(log_price_forecast_df)


    # # 5) compute Markowitz weights using forecasted returns
    # weights_arr, _ = _markowitz_from_forecasts(forecast_matrix, objective=objective)
    # weights = pd.Series(weights_arr, index=tickers, name="weight")

    return price_forecast_df, start, end#, weights

In [35]:
start

Timestamp('2023-08-23 00:00:00')

In [36]:
end

Timestamp('2025-02-01 00:00:00')

In [8]:
#forecast_df, weights = forecast_and_markowitz("2025-02-01")
forecast_df, start, end = forecast_and_markowitz("2025-02-01")

print("First 5 of the 60-day forecasts:")
display(forecast_df.head())

print("\nMarkowitz weights from forecasted returns:")
print(weights)

/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/80/nt46s_zj37q09s9bvd_1j0vw0000gn/T/ipykernel_29464/2880841345.py:45: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressi

First 5 of the 60-day forecasts:


,WMT,NVDA,MPC,XOM
0,71.508766,111.544783,142.544835,103.105764
1,71.508766,111.544783,142.544835,103.162381
2,71.508766,111.544783,142.544835,103.217877
3,71.508766,111.544783,142.544835,103.272273
4,71.508766,111.544783,142.544835,103.325592



Markowitz weights from forecasted returns:


NameError: name 'weights' is not defined

In [18]:
mu = mean_historical_return(forecast_df, log_returns=True)
Sigma = CovarianceShrinkage(forecast_df, log_returns=True).ledoit_wolf()

# Compute tangency portfolio
ef_tan = EfficientFrontier(mu, Sigma)
tangency_portfolio = ef_tan.max_sharpe(risk_free_rate=0.02)

weights_df = pd.DataFrame(
    [(stock, weight) for stock, weight in ef_tan.clean_weights().items()],
    columns=['Stock', 'Weight']
)

weights_df

,Stock,Weight
0,WMT,0.0
1,NVDA,0.0
2,MPC,0.0
3,XOM,1.0
